# VQA 최적화 솔루션 — InternVL2-8B (로컬)

| 항목 | 사양 |
|------|------|
| 모델 | `OpenGVLab/InternVL2-8B` (~8B) |
| 이미지 크기 | **448×448** (InternVL 기본 권장 해상도) |
| 이미지 전처리 | ImageNet 정규화 + **학습 시 증강** |
| 증강 | Flip / ColorJitter / Rotation / RandomAffine |
| 학습 | LoRA r=16, 1 Epoch, Cosine LR, Gradient Clipping |
| 레이블 마스킹 | 답변 토큰만 loss 계산 |
| 추론 | **Logit 직접 비교** (a/b/c/d 확률) |
| 학습 데이터 | **전체 train.csv** |

> **폴더 구조 가정:**
> ```
> (작업 폴더)/
>   train.csv
>   test.csv
>   train/   ← 학습 이미지
>   test/    ← 테스트 이미지
> ```

## 1. 라이브러리 설치

In [ ]:
!pip install -q \
    "transformers>=4.43.2,<5.0.0" \
    "accelerate>=0.34.2" \
    "peft>=0.13.2" \
    "bitsandbytes>=0.43.3" \
    einops timm sentencepiece tiktoken

## 2. CUDA 환경 확인

In [ ]:
import torch
print("PyTorch 버전:", torch.__version__)
print("CUDA 사용 가능:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name())
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print("bfloat16 지원:", torch.cuda.is_bf16_supported())

## 3. 라이브러리 임포트 & 하이퍼파라미터

In [ ]:
import os, math, random
from dataclasses import dataclass
from typing import Any, List

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm.auto import tqdm
from transformers import (
    AutoModel,
    AutoTokenizer,
    BitsAndBytesConfig,
    get_cosine_schedule_with_warmup,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

Image.MAX_IMAGE_PIXELS = None
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# GPU에 따라 compute dtype 자동 선택 (bfloat16 지원 시 → bfloat16, 아니면 → float16)
COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print("Compute dtype:", COMPUTE_DTYPE)

# ── 하이퍼파라미터 ──────────────────────────────────────────────────────────────
MODEL_ID       = "OpenGVLab/InternVL2-8B"
IMAGE_SIZE     = 448      # InternVL 권장 해상도 (28×28 패치 기준)
NUM_EPOCHS     = 1
BATCH_SIZE     = 1
GRAD_ACCUM     = 4        # 유효 배치 = 4
LR             = 1e-4
MAX_GRAD_NORM  = 1.0
LORA_R         = 16
LORA_ALPHA     = 32
WARMUP_RATIO   = 0.05
SEED           = 42

# ── 경로 설정 (모든 파일이 같은 폴더에 있다고 가정) ────────────────────────────
BASE_DIR      = "."   # 필요시 절대 경로로 변경: BASE_DIR = r"C:\your\path"
DATA_PATH     = BASE_DIR
SAVE_DIR      = os.path.join(BASE_DIR, "internvl2_vqa_lora")
os.makedirs(SAVE_DIR, exist_ok=True)

# ImageNet 정규화 (InternVL2 표준)
IMAGENET_MEAN  = (0.485, 0.456, 0.406)
IMAGENET_STD   = (0.229, 0.224, 0.225)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

## 4. 데이터 로드

In [ ]:
def fix_image_path(p: str) -> str:
    """CSV의 상대 경로를 BASE_DIR 기준 절대 경로로 변환"""
    p = str(p).replace('\\', '/')
    if os.path.isabs(p):
        return p  # 이미 절대 경로면 그대로
    return os.path.join(BASE_DIR, p)


train_df = pd.read_csv(os.path.join(DATA_PATH, "train.csv"))
test_df  = pd.read_csv(os.path.join(DATA_PATH, "test.csv"))

train_df['path'] = train_df['path'].apply(fix_image_path)
test_df['path']  = test_df['path'].apply(fix_image_path)

print(f"Train: {len(train_df)}개 / Test: {len(test_df)}개")
print("\n답변 분포 (train):")
print(train_df["answer"].value_counts().sort_index())

## 5. 이미지 전처리 & 증강

In [ ]:
def get_train_transform(image_size: int = IMAGE_SIZE):
    """학습용: 증강 + 정규화 (PIL → Tensor)"""
    return transforms.Compose([
        transforms.Resize((image_size, image_size),
                          interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ColorJitter(
            brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05
        ),
        transforms.RandomRotation(degrees=15, fill=128),
        transforms.RandomAffine(
            degrees=0, translate=(0.05, 0.05), fill=128
        ),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])


def get_val_transform(image_size: int = IMAGE_SIZE):
    """추론용: 리사이즈 + 정규화만"""
    return transforms.Compose([
        transforms.Resize((image_size, image_size),
                          interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])


train_transform = get_train_transform()
val_transform   = get_val_transform()
print("학습 transform:", train_transform)

## 6. 모델 & Tokenizer 로드

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    use_fast=False,
)

base_model = AutoModel.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=COMPUTE_DTYPE,
    device_map={"" : 0},
    trust_remote_code=True,
)

base_model = prepare_model_for_kbit_training(base_model)

num_image_token = base_model.num_image_token
img_context_token_id = tokenizer.convert_tokens_to_ids("<IMG_CONTEXT>")
print(f"이미지 토큰 수: {num_image_token}")
print(f"<IMG_CONTEXT> token_id: {img_context_token_id}")
base_model.img_context_token_id = img_context_token_id

# InternVL2-8B는 InternLM2 백본 사용 → wqkv/wo/w1/w2/w3 구조
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    bias="none",
    target_modules=["wqkv", "wo", "w1", "w2", "w3"],
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

## 7. 프롬프트 & 대화 포맷

In [ ]:
SYSTEM_INSTRUCT = (
    "You are an expert in visual analysis and recycling classification. "
    "Carefully observe the given image and prioritize the following visual elements before making a decision:\n"
    "1. Transparency and light reflection (to distinguish glass from clear plastic).\n"
    "2. Surface wrinkles, gloss, and texture (to distinguish vinyl/plastic wrap from paper).\n"
    "3. The structural shape, quantity of objects, and any printed text/labels."
)

TASK_INSTRUCT = (
    "Based on the visual analysis and the question, select the most appropriate answer. "
    "Do NOT provide any explanations, reasoning, or punctuation. "
    "You must output exactly one lowercase letter: a, b, c, or d.\n\n"
)

IMG_START = "<img>"
IMG_END   = "</img>"
IMG_CTX   = "<IMG_CONTEXT>"


def build_mc_prompt(question: str, a: str, b: str, c: str, d: str) -> str:
    return (
        TASK_INSTRUCT +
        f"{question}\n"
        f"(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}"
    )


def build_conversation_text(
    question: str, a: str, b: str, c: str, d: str,
    answer: str = None
) -> str:
    """InternVL2 chat template 형식으로 대화 텍스트 생성"""
    image_tokens = IMG_START + IMG_CTX * num_image_token + IMG_END
    user_content = image_tokens + "\n" + build_mc_prompt(question, a, b, c, d)

    conversation = [
        {"role": "system", "content": SYSTEM_INSTRUCT},
        {"role": "user",   "content": user_content},
    ]
    if answer is not None:
        conversation.append({"role": "assistant", "content": answer})

    return tokenizer.apply_chat_template(
        conversation,
        tokenize=False,
        add_generation_prompt=(answer is None),
    )

## 8. Dataset & DataCollator

In [ ]:
CHOICES = ["a", "b", "c", "d"]


class VQAMCDataset(Dataset):
    def __init__(self, df, train: bool = True):
        self.df        = df.reset_index(drop=True)
        self.train     = train
        self.transform = train_transform if train else val_transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row          = self.df.iloc[i]
        img          = Image.open(row["path"]).convert("RGB")
        pixel_values = self.transform(img)
        answer       = str(row["answer"]).strip().lower() if self.train else None
        text         = build_conversation_text(
            str(row["question"]), str(row["a"]), str(row["b"]),
            str(row["c"]),        str(row["d"]), answer=answer
        )
        return {"text": text, "pixel_values": pixel_values, "answer": answer}


@dataclass
class DataCollatorV2:
    tokenizer: Any
    train: bool = True

    def __call__(self, batch: List[dict]):
        texts        = [s["text"] for s in batch]
        pixel_values = torch.stack([s["pixel_values"] for s in batch])

        enc = self.tokenizer(
            texts, padding=True, truncation=True,
            max_length=2048, return_tensors="pt",
        )

        result = {
            "input_ids":      enc["input_ids"],
            "attention_mask": enc["attention_mask"],
            "pixel_values":   pixel_values,
            "image_flags":    torch.ones(len(batch), 1, dtype=torch.long),
        }

        if self.train:
            labels = enc["input_ids"].clone()

            for i, sample in enumerate(batch):
                answer   = sample["answer"]
                full_ids = enc["input_ids"][i]
                real_len = enc["attention_mask"][i].sum().item()

                ans_id = self.tokenizer.encode(
                    answer, add_special_tokens=False
                )[0]

                ans_pos = -1
                for pos in range(real_len - 1, -1, -1):
                    if full_ids[pos].item() == ans_id:
                        ans_pos = pos
                        break

                labels[i, :] = -100
                if ans_pos >= 0:
                    labels[i, ans_pos] = full_ids[ans_pos]

            result["labels"] = labels

        return result

## 9. DataLoader

In [ ]:
train_ds = VQAMCDataset(train_df, train=True)

collator_train = DataCollatorV2(tokenizer, train=True)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=collator_train, num_workers=0, pin_memory=False
)
print(f"Train 배치 수: {len(train_loader)}")

## 10. Logit 추론 함수

In [ ]:
# a, b, c, d 토큰 ID
choice_token_ids = []
for ch in CHOICES:
    ids = tokenizer.encode(ch, add_special_tokens=False)
    choice_token_ids.append(ids[0])
    print(f"'{ch}' → token_id = {ids[0]}")
choice_ids_tensor = torch.tensor(choice_token_ids, device=device)


@torch.no_grad()
def predict_logit(img: Image.Image, question: str,
                  a: str, b: str, c: str, d: str) -> str:
    """이미지 + 질문을 받아 a/b/c/d 로짓 비교로 정답 반환"""
    pv   = val_transform(img).unsqueeze(0).to(device)
    text = build_conversation_text(question, a, b, c, d, answer=None)
    enc  = tokenizer(
        text, return_tensors="pt", truncation=True, max_length=2048
    ).to(device)

    image_flags = torch.ones(1, 1, dtype=torch.long).to(device)
    with torch.amp.autocast("cuda", dtype=COMPUTE_DTYPE):
        out = model(
            input_ids=enc["input_ids"],
            attention_mask=enc["attention_mask"],
            pixel_values=pv,
            image_flags=image_flags,
        )

    last_logits = out.logits[0, -1, choice_ids_tensor]
    return CHOICES[last_logits.argmax().item()]

## 11. Fine-tuning

In [ ]:
num_update_steps = NUM_EPOCHS * math.ceil(len(train_loader) / GRAD_ACCUM)
num_warmup_steps = int(num_update_steps * WARMUP_RATIO)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = get_cosine_schedule_with_warmup(
    optimizer, num_warmup_steps=num_warmup_steps, num_training_steps=num_update_steps
)
scaler = torch.amp.GradScaler("cuda", enabled=(COMPUTE_DTYPE == torch.float16))

for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0.0
    optimizer.zero_grad(set_to_none=True)

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [train]")
    for step, batch in enumerate(pbar, start=1):
        batch = {k: v.to(device) for k, v in batch.items()}

        with torch.amp.autocast("cuda", dtype=COMPUTE_DTYPE):
            out  = model(**batch)
            loss = out.loss / GRAD_ACCUM

        scaler.scale(loss).backward()
        running_loss += loss.item()

        if step % GRAD_ACCUM == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

            avg = running_loss / GRAD_ACCUM
            pbar.set_postfix({"loss": f"{avg:.4f}",
                              "lr":   f"{scheduler.get_last_lr()[0]:.2e}"})
            running_loss = 0.0

    model.save_pretrained(SAVE_DIR)
    tokenizer.save_pretrained(SAVE_DIR)
    print(f"  ✓ Epoch {epoch+1} model saved → {SAVE_DIR}")

print(f"\n학습 완료.")

## 12. 테스트 추론 & 제출

In [ ]:
model.eval()
preds = []

for i in tqdm(range(len(test_df)), desc="Test Inference"):
    row  = test_df.iloc[i]
    img  = Image.open(row["path"]).convert("RGB")
    pred = predict_logit(
        img, str(row["question"]),
        str(row["a"]), str(row["b"]), str(row["c"]), str(row["d"])
    )
    preds.append(pred)

print("\n예측 분포:")
print(pd.Series(preds).value_counts().sort_index())

submission = pd.DataFrame({"id": test_df["id"], "answer": preds})
submission.to_csv(os.path.join(BASE_DIR, "submission.csv"), index=False)
print(f"Saved: {os.path.join(BASE_DIR, 'submission.csv')}")
submission.head(10)